In [142]:
%pip install -q --progress-bar off --disable-pip-version-check klotho-cac supriya
import random

from itertools import cycle

from klotho import play, plot, set_audio_engine, register_synthdef

from klotho.chronos import Meas, TemporalUnitSequence as UTS, TemporalBlock as BT

from klotho.thetos import (CompositionalUnit as UC, Ensemble, Score,
                           SynthDefFX as InsertFX)

from klotho.tonos import Scale, Chord, Pitch, ChordSequence

from klotho.topos import Pattern

from klotho.dynatos import Envelope as Env, map_curve

from supriya import synthdef, Envelope, DoneAction

from supriya.ugens import (SinOsc, Saw, Pulse, WhiteNoise, PinkNoise, Impulse,
                           EnvGen, Pan2, Out, LPF, HPF, BPF, RLPF, Lag, LeakDC,
                           LFNoise2, Pluck, Ringz)

set_audio_engine("supersonic")

In [143]:
# @title
@synthdef()
def muse_fuzzbass(out=0, freq=110.0, amp=0.5, gate=1, pan=0.0,
                  fuzz=8.0, octave=0.4, sub=0.6, toneLPFRatio=4.0, rq=0.45,
                  portamento=0.02,
                  attackTime=0.008, decayTime=0.15, sustainLevel=0.8,
                  releaseTime=0.12, peakLevel=1.0,
                  attackCurve=-4.0, decayCurve=-2.0, releaseCurve=-4.0):
    # portamento = a short glide into each new pitch, like sliding on a fretboard
    f = Lag.kr(source=freq.max(0.001), lag_time=portamento.max(0.0))

    # the ADSR. release_node=2 marks which point the `gate` releases from.
    env = EnvGen.kr(
        envelope=Envelope(
            amplitudes=[0, peakLevel, sustainLevel * peakLevel, 0],
            durations=[attackTime, decayTime, releaseTime],
            curves=[attackCurve, decayCurve, releaseCurve],
            release_node=2,
        ),
        gate=gate, done_action=DoneAction.FREE_SYNTH,
    )

    core = Saw.ar(frequency=f) + Pulse.ar(frequency=f * 1.005, width=0.42)
    core = core + (Saw.ar(frequency=f * 2) * octave.clip(0.0, 1.0))   # octave doubling
    dirty = (core * fuzz.max(1.0)).tanh()                             # <- the fuzz pedal

    cutoff = (f * toneLPFRatio.max(0.5) * (1 + env)).clip(60, 5000)   # filter follows env
    sig = RLPF.ar(source=dirty, frequency=cutoff, reciprocal_of_q=rq.clip(0.05, 2.0))

    # the clean sub, blended in AFTER the fuzz so the distortion cannot squash it
    weight = LPF.ar(source=SinOsc.ar(frequency=f), frequency=200) * sub.clip(0.0, 1.5) * 1.2

    sig = (sig + weight) * env * amp * 0.3
    Out.ar(bus=out, source=Pan2.ar(source=sig, position=pan))

In [144]:
# @title
PIANO_PARTIALS = ((1, 1.00, 1.00), (2, 0.52, 0.62), (3, 0.30, 0.44),
                  (4, 0.17, 0.31), (5, 0.10, 0.22), (6, 0.06, 0.16))

@synthdef()
def muse_piano(out=0, freq=440.0, amp=0.5, gate=1, pan=0.0,
               brightness=0.5, strike=0.6, detune=0.45, inharm=0.0007,
               decayTime=2.6, attackTime=0.003, releaseTime=0.9,
               releaseCurve=-4.0):
    f = freq.max(0.001)

    # The damper. It does NOT shape the note's decay (the partials do that) -- it just
    # cuts the string off when the key is released, like a real damper falling.
    damper = EnvGen.kr(
        envelope=Envelope(amplitudes=[0, 1, 1, 0],
                          durations=[attackTime, 0.1, releaseTime],
                          curves=[0, 0, releaseCurve], release_node=2),
        gate=gate, done_action=DoneAction.FREE_SYNTH)

    # --- the strings: every partial gets its OWN decay ---
    sig = None
    for n, level, dfac in PIANO_PARTIALS:
        fn = f * n * (1 + inharm * (n * n)).sqrt()      # inharmonicity: stretched sharp
        penv = EnvGen.ar(envelope=Envelope.percussive(0.002, decayTime * dfac, curve=-4.0))
        lvl = level * (1 + brightness * min(n - 1, 4) * 0.22)
        part = SinOsc.ar(frequency=fn) * penv * lvl
        sig = part if sig is None else sig + part

    # --- two strings per key, a hair apart -> the shimmering tail ---
    d = detune * 0.0009
    for mul in (1 + d, 1 - d):
        penv = EnvGen.ar(envelope=Envelope.percussive(0.002, decayTime, curve=-4.0))
        sig = sig + SinOsc.ar(frequency=f * mul) * penv * 0.5

    # --- the hammer: a bright knock plus a soft low thud ---
    knock = HPF.ar(source=WhiteNoise.ar(), frequency=2200) * EnvGen.ar(
        envelope=Envelope.percussive(0.0003, 0.011, curve=-8.0)) * strike
    thud = LPF.ar(source=PinkNoise.ar(), frequency=320) * EnvGen.ar(
        envelope=Envelope.percussive(0.0006, 0.045, curve=-6.0)) * strike * 0.7

    sig = LPF.ar(source=sig + knock + thud, frequency=(f * 6 + 1500).clip(600, 12000))
    sig = sig * damper * amp * 0.16
    Out.ar(bus=out, source=Pan2.ar(source=sig, position=pan))

In [145]:
# @title
@synthdef()
def muse_leadgtr(out=0, freq=440.0, amp=0.4, gate=1, pan=0.0,
                 drive=18.0, cab=4500.0, presence=0.8, bite=0.6, sag=0.15,
                 pickup=2600.0, pickupQ=0.55, pick=0.5,
                 vibRate=5.5, vibDepth=0.0, portamento=0.03,
                 attackTime=0.012, decayTime=0.25, sustainLevel=0.85,
                 releaseTime=0.28, peakLevel=1.0,
                 attackCurve=-4.0, decayCurve=-2.0, releaseCurve=-4.0):
    vib = SinOsc.kr(frequency=vibRate.max(0.01)) * vibDepth * 0.06 + 1.0   # whammy wobble
    f = Lag.kr(source=freq.max(0.001), lag_time=portamento.max(0.0)) * vib
    env = EnvGen.kr(
        envelope=Envelope(
            amplitudes=[0, peakLevel, sustainLevel * peakLevel, 0],
            durations=[attackTime, decayTime, releaseTime],
            curves=[attackCurve, decayCurve, releaseCurve],
            release_node=2,
        ),
        gate=gate, done_action=DoneAction.FREE_SYNTH,
    )

    # --- the string: two detuned saws + a pulse = a double-tracked guitar ---
    core = (Saw.ar(frequency=f)
            + Saw.ar(frequency=f * 1.006) * 0.7
            + Pulse.ar(frequency=f * 0.996, width=0.32) * 0.5)

    # the pick striking the string: short, bright, and it hits the preamp too
    strike = HPF.ar(source=WhiteNoise.ar(), frequency=1800) * EnvGen.ar(
        envelope=Envelope.percussive(0.001, 0.014, curve=-8.0)) * pick.clip(0.0, 2.0)

    # --- 1. PICKUP: a RESONANT peak, not a plain rolloff. This is the electric part. ---
    pre = RLPF.ar(source=core + strike, frequency=pickup.clip(600, 6000),
                  reciprocal_of_q=pickupQ.clip(0.2, 1.5))

    # --- 2. PRE-GAIN EQ: cut the mud, push the mids ---
    pre = HPF.ar(source=pre, frequency=170)
    pre = pre + BPF.ar(source=pre, frequency=880, reciprocal_of_q=0.8) * bite

    # --- 3. TWO CASCADED GAIN STAGES; `sag` makes stage 2 asymmetric (even harmonics) ---
    s1 = (pre * drive.max(1.0)).tanh()
    s2 = LeakDC.ar(source=(s1 * 2.2 + sag).tanh())

    # --- 4. SPEAKER CABINET: ONE pole, plus a presence peak. Two poles = lifeless. ---
    c = LPF.ar(source=s2, frequency=cab.clip(1200, 8000))
    c = LPF.ar(source=c, frequency=7500)                    # trim true fizz only
    c = HPF.ar(source=c, frequency=95)
    c = c + BPF.ar(source=c, frequency=3000, reciprocal_of_q=0.6) * presence

    Out.ar(bus=out, source=Pan2.ar(source=c * env * amp * 0.18, position=pan))

@synthdef()
def muse_pickgtr(out=0, freq=330.0, amp=0.5, gate=1, pan=0.0,
                 pick=0.7, pickTone=2600.0, damp=0.45, ring=2.6, body=0.35,
                 drive=2.4, cab=4200.0, presence=0.5,
                 attackTime=0.001, decayTime=0.05, sustainLevel=1.0,
                 releaseTime=1.4, peakLevel=1.0,
                 attackCurve=-4.0, decayCurve=-2.0, releaseCurve=-3.0):
    f = freq.max(20.0)

    # THE PLECTRUM: a few milliseconds of filtered noise -- the pick you actually hear.
    exc = LPF.ar(source=WhiteNoise.ar(), frequency=pickTone.clip(400, 9000))
    exc = exc * EnvGen.ar(envelope=Envelope.percussive(0.0004, 0.006, curve=-8.0))
    exc = exc * pick.clip(0.0, 2.0)

    # KARPLUS-STRONG: run that burst round a delay line exactly one wavelength long,
    # losing a little top end each lap -> a real decaying string.
    string = Pluck.ar(source=exc, trigger=Impulse.kr(frequency=0),
                      maximum_delay_time=0.05,
                      delay_time=f.reciprocal().clip(0.0005, 0.049),
                      decay_time=ring.max(0.05),
                      coefficient=damp.clip(0.05, 0.95))

    bod = Ringz.ar(source=exc, frequency=200, decay_time=0.09) * body.clip(0, 1)

    env = EnvGen.kr(
        envelope=Envelope(amplitudes=[0, peakLevel, sustainLevel * peakLevel, 0],
                          durations=[attackTime, decayTime, releaseTime],
                          curves=[attackCurve, decayCurve, releaseCurve],
                          release_node=2),
        gate=gate, done_action=DoneAction.FREE_SYNTH)

    pre = HPF.ar(source=string + bod * 0.3, frequency=110)
    amped = (pre * drive.max(1.0)).tanh()                    # light overdrive only
    c2 = LPF.ar(source=amped, frequency=cab.clip(1200, 9000))
    c2 = HPF.ar(source=c2, frequency=95)
    c2 = c2 + BPF.ar(source=c2, frequency=3000, reciprocal_of_q=0.7) * presence
    Out.ar(bus=out, source=Pan2.ar(source=c2 * env * amp * 0.35, position=pan))

@synthdef()
def muse_strings(out=0, freq=440.0, amp=0.35, gate=1, pan=0.0,
                 spread=0.7, toneLPFRatio=4.0, bowNoise=0.05,
                 attackTime=0.5, decayTime=0.4, sustainLevel=0.85,
                 releaseTime=1.2, peakLevel=1.0,
                 attackCurve=2.0, decayCurve=-2.0, releaseCurve=-3.0):
    f = freq.max(0.001)
    drift = LFNoise2.kr(frequency=0.3) * spread * 0.004 + 1.0   # players drifting apart
    env = EnvGen.kr(
        envelope=Envelope(
            amplitudes=[0, peakLevel, sustainLevel * peakLevel, 0],
            durations=[attackTime, decayTime, releaseTime],
            curves=[attackCurve, decayCurve, releaseCurve],
            release_node=2,
        ),
        gate=gate, done_action=DoneAction.FREE_SYNTH,
    )
    core = (Saw.ar(frequency=f * drift)
            + Saw.ar(frequency=f * 1.004) * 0.8
            + Saw.ar(frequency=f * 0.996) * 0.8)
    core = core + PinkNoise.ar() * bowNoise.clip(0.0, 1.0)
    sig = LPF.ar(source=core, frequency=(f * toneLPFRatio.max(1.0)).clip(300, 9000))
    sig = sig * env * amp * 0.2
    Out.ar(bus=out, source=Pan2.ar(source=sig, position=pan))

In [146]:
# @title
@synthdef()
def muse_kick(out=0, freq=52.0, amp=0.9, pan=0.0, velocity=1.0,
              pitchDrop=6.0, pitchDropTime=0.035, clickAmp=0.3, releaseTime=0.42):
    # pitch envelope: start 6x above the target and fall onto it in 35 ms
    fenv = EnvGen.kr(envelope=Envelope(amplitudes=[freq * pitchDrop, freq],
                                       durations=[pitchDropTime], curves=[-6.0]))
    aenv = EnvGen.ar(envelope=Envelope.percussive(0.001, releaseTime, curve=-5.0),
                     done_action=DoneAction.FREE_SYNTH)
    body = SinOsc.ar(frequency=fenv)
    click = HPF.ar(source=WhiteNoise.ar(), frequency=1200) * EnvGen.ar(
        envelope=Envelope.percussive(0.0005, 0.02, curve=-8.0)) * clickAmp.clip(0.0, 1.0)
    sig = ((body + click) * 1.4).tanh() * aenv * amp * velocity * 0.7
    Out.ar(bus=out, source=Pan2.ar(source=sig, position=pan))

@synthdef()
def muse_snare(out=0, freq=190.0, amp=0.7, pan=0.0, velocity=1.0,
               snap=0.7, tone=0.5, releaseTime=0.28):
    aenv = EnvGen.ar(envelope=Envelope.percussive(0.001, releaseTime, curve=-4.0),
                     done_action=DoneAction.FREE_SYNTH)
    body = (SinOsc.ar(frequency=freq) + SinOsc.ar(frequency=freq * 1.6) * 0.6) * tone.clip(0.0, 1.0)
    body = body * EnvGen.ar(envelope=Envelope.percussive(0.001, 0.09, curve=-5.0))
    noise = HPF.ar(source=WhiteNoise.ar(), frequency=900) * snap.clip(0.0, 1.0)
    noise = noise * EnvGen.ar(envelope=Envelope.percussive(0.001, releaseTime, curve=-3.0))
    sig = ((body + noise) * 1.6).tanh() * aenv * amp * velocity * 0.5
    Out.ar(bus=out, source=Pan2.ar(source=sig, position=pan))

@synthdef()
def muse_hat(out=0, amp=0.4, pan=0.0, velocity=1.0, tone=7000.0, releaseTime=0.06):
    aenv = EnvGen.ar(envelope=Envelope.percussive(0.0005, releaseTime, curve=-6.0),
                     done_action=DoneAction.FREE_SYNTH)
    sig = HPF.ar(source=WhiteNoise.ar(), frequency=tone.clip(1000, 14000))
    Out.ar(bus=out, source=Pan2.ar(source=sig * aenv * amp * velocity * 0.3, position=pan))

@synthdef()
def muse_crash(out=0, amp=0.5, pan=0.0, velocity=1.0, tone=4000.0, releaseTime=2.0):
    aenv = EnvGen.ar(envelope=Envelope.percussive(0.002, releaseTime, curve=-3.0),
                     done_action=DoneAction.FREE_SYNTH)
    sig = HPF.ar(source=WhiteNoise.ar() + PinkNoise.ar(), frequency=tone.clip(500, 12000))
    Out.ar(bus=out, source=Pan2.ar(source=sig * aenv * amp * velocity * 0.2, position=pan))

@synthdef()
def muse_ride(out=0, amp=0.4, pan=0.0, velocity=1.0,
              ping=0.7, tone=6500.0, releaseTime=1.7):
    # A cymbal is a metal plate ringing at unrelated frequencies. Noise alone is a
    # hiss; Ringz resonators at inharmonic pitches add the metallic 'ping'.
    exc = WhiteNoise.ar() * EnvGen.ar(
        envelope=Envelope.percussive(0.0005, 0.004, curve=-8.0))
    metal = (Ringz.ar(source=exc, frequency=2100, decay_time=releaseTime * 0.5)
             + Ringz.ar(source=exc, frequency=3170, decay_time=releaseTime * 0.4)
             + Ringz.ar(source=exc, frequency=4790, decay_time=releaseTime * 0.3))
    wash = HPF.ar(source=WhiteNoise.ar(), frequency=tone.clip(1000, 14000))
    wash = wash * EnvGen.ar(envelope=Envelope.percussive(0.001, releaseTime, curve=-3.5))
    aenv = EnvGen.ar(envelope=Envelope.percussive(0.001, releaseTime * 1.15, curve=-3.0),
                     done_action=DoneAction.FREE_SYNTH)
    sig = (metal * ping.clip(0, 2) * 0.3 + wash * 0.45) * aenv * amp * velocity * 0.22
    Out.ar(bus=out, source=Pan2.ar(source=sig, position=pan))

@synthdef()
def muse_tom(out=0, freq=160.0, amp=0.7, pan=0.0, velocity=1.0,
             pitchDrop=1.9, pitchDropTime=0.07, skin=0.35, releaseTime=0.55):
    fenv = EnvGen.kr(envelope=Envelope(amplitudes=[freq * pitchDrop, freq],
                                       durations=[pitchDropTime], curves=[-4.0]))
    aenv = EnvGen.ar(envelope=Envelope.percussive(0.001, releaseTime, curve=-4.0),
                     done_action=DoneAction.FREE_SYNTH)
    body = SinOsc.ar(frequency=fenv) + SinOsc.ar(frequency=fenv * 1.5) * 0.3
    skn = HPF.ar(source=WhiteNoise.ar(), frequency=600) * EnvGen.ar(
        envelope=Envelope.percussive(0.0005, 0.03, curve=-7.0)) * skin.clip(0, 1)
    sig = ((body + skn) * 1.2).tanh() * aenv * amp * velocity * 0.5
    Out.ar(bus=out, source=Pan2.ar(source=sig, position=pan))

In [147]:
for d in [muse_fuzzbass, muse_piano, muse_leadgtr, muse_pickgtr, muse_strings,
          muse_kick, muse_snare, muse_hat, muse_crash, muse_ride, muse_tom]:
    register_synthdef(d)

ens = Ensemble('muse', families={
    'drums':   {'kick': 'muse_kick', 'snare': 'muse_snare', 'hat': 'muse_hat',
                'crash': 'muse_crash', 'ride': 'muse_ride', 'tom': 'muse_tom'},
    'bass':    {'fuzz': 'muse_fuzzbass'},
    'guitar':  {'lead': 'muse_leadgtr', 'pick': 'muse_pickgtr'},
    'keys':    {'piano': 'muse_piano'},
    'strings': {'pad': 'muse_strings'},
})

In [148]:
QUALITIES = {
    'min':  ['0', '300', '700'],           'maj':  ['0', '400', '700'],
    'dim':  ['0', '300', '600'],           'aug':  ['0', '400', '800'],
    'sus4': ['0', '500', '700'],           '5':    ['0', '700'],          # power chord
    'min7': ['0', '300', '700', '1000'],   'dom7': ['0', '400', '700', '1000'],
}

NATURAL_MINOR    = ['0', '200', '300', '500', '700', '800', '1000']

HARMONIC_MINOR   = ['0', '200', '300', '500', '700', '800', '1100']   # raised 7th

PHRYGIAN_DOMINANT = ['0', '100', '400', '500', '700', '800', '1000']  # the "Muse" scale

def chord12(root, quality='min'):
    '''A 12-tone-equal-tempered chord on `root`, e.g. chord12("D3", "min").'''
    return Chord(QUALITIES[quality], interval_type='cents').root(root)

def scale12(root, shape=NATURAL_MINOR):
    '''A 12-TET scale on `root`.'''
    return Scale(shape, interval_type='cents').root(root)

In [149]:
PROGRESSIONS = {
    'andalusian': [chord12('D3', 'min'), chord12('C3', 'maj'),
                   chord12('Bb2', 'maj'), chord12('A2', 'maj')],
    'atheist':    [chord12('D3', 'min'), chord12('D3', 'min'),
                   chord12('Bb2', 'maj'), chord12('C3', 'maj')],
    'escape':     [chord12('D3', 'min'), chord12('A2', 'min'),
                   chord12('Bb2', 'maj'), chord12('F2', 'maj')],
    'descent':    [chord12('D3', 'min'), chord12('C#3', 'dim'),
                   chord12('C3', 'maj'), chord12('B2', 'dim')],
}

PROG = PROGRESSIONS['andalusian'] # default

LEAD_SCALE = scale12('D4', HARMONIC_MINOR)

SECTION_PROGS = {
    # i - bVI - bIII - bVII
    'intro':     [chord12('D3', 'min'), chord12('Bb2', 'maj'),
                  chord12('F3', 'maj'), chord12('C3', 'maj')],

    # i - bVII - bVI - V
    'verse':     PROGRESSIONS['andalusian'],

    # one chord, held as a pedal while everything swells over it
    'build':     [chord12('D3', 'min')],

    # bVI - bVII - i - i
    'chorus':    [chord12('Bb2', 'maj'), chord12('C3', 'maj'),
                  chord12('D3', 'min'),  chord12('D3', 'min')],

    # i - V - bVI - bIII
    'breakdown': [chord12('D3', 'min'), chord12('A2', 'maj'),
                  chord12('Bb2', 'maj'), chord12('F3', 'maj')],

    # i - bVI - i
    'outro':     [chord12('D3', 'min'), chord12('Bb2', 'maj'), chord12('D3', 'min')],
}

In [150]:
TEMPUS = Meas('4/4')

BEAT   = '1/4'

BPM    = 138

TIME = dict(tempus=TEMPUS, beat=BEAT, bpm=BPM)

In [151]:
def make_drums(n_bars, span=1, tempus=TEMPUS, beat=BEAT, bpm=BPM, amp=1.0,
               kick_pat=(1, -1, (1, (-1, 1)), -1), snare_pat=(-1, 1, -1, 1),
               n_hats=8, crash=False, **kw):
    hats, kicks, snares = UTS(), UTS(), UTS()
    for _ in range(n_bars):
        h = UC(span=span, tempus=tempus, prolatio=(1,) * n_hats, beat=beat, bpm=bpm,
               inst=ens.drums['hat'])
        h.leaves.set(amp=lambda: random.uniform(0.5, 0.9) * amp, **kw)
        hats.append(h)

        k = UC(span=span, tempus=tempus, prolatio=kick_pat, beat=beat, bpm=bpm,
               inst=ens.drums['kick'])
        k.leaves.set(amp=amp, **kw)
        kicks.append(k)

        s = UC(span=span, tempus=tempus, prolatio=snare_pat, beat=beat, bpm=bpm,
               inst=ens.drums['snare'])
        s.leaves.set(amp=0.9 * amp, **kw)
        snares.append(s)

    rows = [hats, kicks, snares]
    if crash:
        c = UC(span=span * n_bars, tempus=tempus, prolatio=(1, -3), beat=beat, bpm=bpm,
               inst=ens.drums['crash'])
        c.leaves.set(amp=0.7 * amp)
        rows.append(c)
    return BT(rows, axis=-1)

In [152]:
def make_fuzz_riff(chords, span=1, tempus=TEMPUS, beat=BEAT, bpm=BPM,
                   degrees=(0, 0, 3, 0, 0, 3, -1, 0), n_pulses=8, amp=0.85, **kw):
    seq = UTS()
    for ch in chords:
        uc = UC(span=span, tempus=tempus, prolatio=(1,) * n_pulses, beat=beat, bpm=bpm,
                inst=ens.bass['fuzz'])
        pat = Pattern(list(degrees))
        uc.leaves.set(freq=lambda ch=ch, pat=pat: ch[next(pat)].freq, amp=amp, **kw)
        seq.append(uc)
    return seq

In [153]:
def make_piano_arp(chords, span=1, tempus=TEMPUS, beat=BEAT, bpm=BPM,
                   degrees=(0, 1, 2, 3, 4, 3, 2, 1), prolatio=((1, (1, 1, 1)),) * 4,
                   amp=0.55, shift=1, **kw):
    seq = UTS()
    for ch in chords:
        c = ch.equave_shift(shift)
        uc = UC(span=span, tempus=tempus, prolatio=prolatio, beat=beat, bpm=bpm,
                inst=ens.keys['piano'])
        pat = Pattern(list(degrees))
        uc.leaves.set(freq=lambda c=c, pat=pat: c[next(pat)].freq, amp=amp, **kw)
        seq.append(uc)
    return seq

In [154]:
def make_power_chords(chords, span=1, tempus=TEMPUS, beat=BEAT, bpm=BPM,
                      prolatio=(1, -1, 1, -1), amp=0.6, shift=1, **kw):
    seq = UTS()
    for ch in chords:
        v = ch.voicing([0, 2]).equave_shift(shift)          # root + fifth = power chord
        uc = UC(span=span, tempus=tempus, prolatio=prolatio, beat=beat, bpm=bpm,
                inst=ens.guitar['lead'])
        uc.leaves.set(freq=tuple(v.pitches), amp=amp, **kw)  # tuple => sounded together
        seq.append(uc)
    return seq

def make_lead(scale, n_bars, span=1, tempus=TEMPUS, beat=BEAT, bpm=BPM,
              degrees=(7, 6, 4, 6, 7, 9, 7, 6), prolatio=(2, 1, 1, (2, (1, 1))),
              amp=0.5, **kw):
    seq = UTS()
    pat = Pattern(list(degrees))
    for _ in range(n_bars):
        uc = UC(span=span, tempus=tempus, prolatio=prolatio, beat=beat, bpm=bpm,
                inst=ens.guitar['lead'])
        uc.leaves.set(freq=lambda: scale[next(pat)].freq, amp=amp, **kw)
        seq.append(uc)
    return seq

def make_picked_arp(chords, span=1, tempus=TEMPUS, beat=BEAT, bpm=BPM,
                    degrees=(0, 1, 2, 3, 2, 1), prolatio=(1,) * 6,
                    amp=0.45, shift=1, **kw):
    # The clean picked-guitar arpeggio: the Thoughts of a Dying Atheist intro texture.
    seq = UTS()
    for ch in chords:
        c = ch.equave_shift(shift)
        uc = UC(span=span, tempus=tempus, prolatio=prolatio, beat=beat, bpm=bpm,
                inst=ens.guitar['pick'])
        pat = Pattern(list(degrees))
        uc.leaves.set(freq=lambda c=c, pat=pat: c[next(pat)].freq, amp=amp, **kw)
        seq.append(uc)
    return seq

def make_strings(chords, span=1, tempus=TEMPUS, beat=BEAT, bpm=BPM,
                 amp=0.5, shift=1, **kw):
    seq = UTS()
    for ch in chords:
        uc = UC(span=span, tempus=tempus, prolatio='d', beat=beat, bpm=bpm,
                inst=ens.strings['pad'])
        uc.root.set(freq=tuple(ch.equave_shift(shift).pitches), amp=amp, **kw)
        seq.append(uc)
    return seq

In [155]:
def _t(time):
    return TIME if time is None else time

def _c(chords, name):
    return SECTION_PROGS[name] if chords is None else chords

def intro_section(chords=None, time=None, piano=0.55, strings=0.50,
                  decay=2.2, strike=0.30, shift=2, **_):
    ch, T = _c(chords, 'intro'), _t(time)
    return BT([
        make_piano_arp(ch, amp=piano, shift=shift, decayTime=decay, strike=strike, **T),
        make_strings(ch, amp=strings, shift=1, attackTime=1.2, releaseTime=2, **T),
    ], axis=-1)

def verse_section(chords=None, time=None, drums=0.75, bass=0.95, piano=0.32,
                  strings=0.34, fuzz=8, octave=0.30, sub=0.60, **_):
    ch, T = _c(chords, 'verse'), _t(time)
    return BT([
        make_drums(len(ch), amp=drums, n_hats=8, **T),
        make_fuzz_riff(ch, amp=bass, fuzz=fuzz, octave=octave, sub=sub, **T),
        make_piano_arp(ch, amp=piano, shift=2, decayTime=1.2, **T),
        make_strings(ch, amp=strings, shift=1, attackTime=1.0, releaseTime=2, **T),
    ], axis=-1)

def build_section(chords=None, time=None, n_bars=4, bass=1.0, drums=0.80,
                  strings=0.55, fuzz=14, octave=0.50, sub=0.70,
                  swell=(0.15, 1.0), **_):
    ch, T = _c(chords, 'build'), _t(time)
    # ONE unit spanning all four bars, so a single envelope can ramp across the lot.
    # (apply_envelope lives on a unit's root, not on a sequence of units.)
    riff = make_fuzz_riff([ch[0]], span=n_bars, n_pulses=8 * n_bars,
                          amp=bass, fuzz=fuzz, octave=octave, sub=sub, **T)
    riff[0].root.apply_envelope(Env(list(swell), curve=2), 'amp')   # quiet -> huge

    drms = make_drums(n_bars, amp=drums, n_hats=16,
                      snare_pat=(-1, 1, -1, (1, (1, 1))), **T)
    return BT([riff, drms,
               make_strings([ch[0]] * 2, span=2, amp=strings, attackTime=2.0, **T)],
              axis=-1)

def chorus_section(chords=None, time=None, drums=1.0, bass=1.0, gtr=0.38, piano=0.30,
                   strings=0.55, fuzz=16, octave=0.60, sub=0.65,
                   drive=18, cab=4200, presence=0.8, **_):
    ch, T = _c(chords, 'chorus'), _t(time)
    return BT([
        make_drums(len(ch), amp=drums, crash=True, n_hats=8, **T),
        make_fuzz_riff(ch, amp=bass, fuzz=fuzz, octave=octave, sub=sub, **T),
        make_power_chords(ch, amp=gtr, prolatio=(1, -1, 1, -1), shift=1,
                          drive=drive, cab=cab, presence=presence, **T),
        make_piano_arp(ch, amp=piano, shift=2, **T),
        make_strings(ch, amp=strings, shift=2, attackTime=0.3, **T),
    ], axis=-1)

def outro_section(chords=None, time=None, piano=(0.55, 0.15), strings=0.40,
                  decay=2.5, strike=0.20, **_):
    ch, T = _c(chords, 'outro'), _t(time)
    # Fade across separate bars: map_curve turns "bar k of n" into an amplitude.
    arp = UTS()
    for k, c in enumerate(ch):
        amp_k = map_curve(k, (0, len(ch) - 1), piano, curve=-2)
        arp.append(make_piano_arp([c], amp=amp_k, shift=2,
                                  decayTime=decay, strike=strike, **T)[0])
    return BT([arp, make_strings(ch, amp=strings, shift=1, attackTime=2.0, **T)],
              axis=-1)

In [156]:
def breakdown_picked(chords=None, time=None, strings=0.70, gtr=0.55,
                     damp=0.5, ring=2.8, pick=0.8, **_):
    # Clean picked arpeggio over strings. You hear every note struck.
    ch, T = _c(chords, 'breakdown'), _t(time)
    return BT([
        make_strings(ch, amp=strings, shift=1, attackTime=1.2, releaseTime=2.8, **T),
        make_picked_arp(ch, amp=gtr, shift=2, damp=damp, ring=ring, pick=pick,
                        degrees=(0, 1, 2, 3, 2, 1), **T),
    ], axis=-1)

def breakdown_piano(chords=None, time=None, strings=0.75, piano=0.55,
                    decay=2.8, **_):
    # No guitar at all. Piano and strings, the air let out of the room.
    ch, T = _c(chords, 'breakdown'), _t(time)
    return BT([
        make_strings(ch, amp=strings, shift=1, attackTime=1.6, releaseTime=3.0, **T),
        make_piano_arp(ch, amp=piano, shift=2, decayTime=decay, strike=0.25,
                       degrees=(0, 2, 4, 2, 1, 2), **T),
    ], axis=-1)

def breakdown_halftime(chords=None, time=None, strings=0.65, bass=0.7, drums=0.5,
                       fuzz=20, octave=0.2, sub=0.9, **_):
    # Slow and heavy: a crawling fuzz bass, toms, no lead. The floor drops out.
    ch, T = _c(chords, 'breakdown'), _t(time)
    toms = UTS()
    for _ch in ch:
        u = UC(span=1, prolatio=(1, -1, 1, -1), inst=ens.drums['tom'], **T)
        u.leaves.set(amp=0.75 * drums)
        toms.append(u)
    return BT([
        make_strings(ch, amp=strings, shift=1, attackTime=1.4, releaseTime=3.0, **T),
        # two slow notes a bar instead of eight -- the same riff at quarter speed
        make_fuzz_riff(ch, amp=bass, fuzz=fuzz, octave=octave, sub=sub,
                       n_pulses=2, degrees=(0, 0), releaseTime=0.6, **T),
        toms,
    ], axis=-1)

def breakdown_lead(chords=None, time=None, scale=None, strings=0.70, lead=0.28,
                   drive=12, cab=3800, presence=0.9, vib=0.6, **_):
    # The original: distorted lead guitar singing over strings.
    ch, T = _c(chords, 'breakdown'), _t(time)
    sc = LEAD_SCALE if scale is None else scale
    return BT([
        make_strings(ch, amp=strings, shift=1, attackTime=1.0, releaseTime=2.5, **T),
        make_lead(sc, len(ch), amp=lead, vibDepth=vib, drive=drive,
                  cab=cab, presence=presence, releaseTime=1.2, **T),
    ], axis=-1)

BREAKDOWNS = {'picked': breakdown_picked, 'piano': breakdown_piano,
              'halftime': breakdown_halftime, 'lead': breakdown_lead}

def breakdown_section(chords=None, time=None, style='picked', **k):
    return BREAKDOWNS[style](chords=chords, time=time, **k)

In [157]:
FINAL_TIME = dict(tempus=Meas('4/4'), beat='1/4', bpm=138)

_DESCENT = [chord12('D3', 'min'), chord12('C#3', 'dim'),
            chord12('C3', 'maj'), chord12('B2', 'dim')]

FINAL_PROGS = {
    'intro':      _DESCENT,
    'verse':      _DESCENT,
    'build':      [chord12('D3', 'min')],
    'breakdown':  [chord12('D3', 'min'), chord12('A2', 'maj'),
                   chord12('Bb2', 'maj'), chord12('F3', 'maj')],
    'chorus':     _DESCENT,
    'build2':     [chord12('D3', 'min')],
    'breakdown2': _DESCENT,
    'outro':      [chord12('D3', 'min'), chord12('C3', 'maj'), chord12('D3', 'min')],
}

FINAL_KNOBS = {
    'intro':      dict(piano=0.45, strings=0.85, decay=3.4, strike=0.18),

    'verse':      dict(drums=0.40, bass=0.45, piano=0.35, strings=0.80,
                       fuzz=14, octave=0.15, sub=0.9),

    'build':      dict(bass=0.90, drums=0.80, strings=0.75,
                       fuzz=16, octave=0.40, sub=0.90, swell=(0.12, 1.0)),

    'breakdown':  dict(style='picked', strings=0.70, gtr=1,
                       damp=0.4, ring=4, pick=0.5),

    'chorus':     dict(drums=0.60, bass=0.55, gtr=0.22, piano=0.28, strings=0.85,
                       fuzz=16, octave=0.30, sub=0.90, drive=12, cab=3400),

    'build2':     dict(bass=0.95, drums=0.85, strings=0.80,
                       fuzz=18, octave=0.45, sub=0.90, swell=(0.08, 1.0)),

    'breakdown2': dict(style='picked', strings=0.85, gtr=1,
                       damp=0.4, ring=4, pick=0.5),

    'outro':      dict(piano=(0.5, 0.08), strings=0.50, decay=3.6),
}

FINAL_ARRANGEMENT = [
    ('intro',      intro_section),
    ('verse',      verse_section),
    ('build',      build_section),
    ('breakdown',  breakdown_section),
    ('chorus',     chorus_section),
    ('build2',     build_section),
    ('breakdown2', breakdown_section),
    ('verse2',     verse_section),
    ('chorus2',    chorus_section),
    ('outro',      outro_section),
]

ens.set_inserts('drums',   [InsertFX('kl_reverb', mix=0.16, room=0.55, damp=0.5)])

ens.set_inserts('bass',    [InsertFX('kl_distortion', dist=0.22, mix=0.35)])

ens.set_inserts('guitar',  [InsertFX('kl_distortion', dist=0.18, mix=0.35),
                            InsertFX('kl_delay', mix=0.22, delayTime=0.217,
                                     decayTime=1.6),
                            InsertFX('kl_reverb', mix=0.25, room=0.7)])

ens.set_inserts('keys',    [InsertFX('kl_reverb', mix=0.32, room=0.8, damp=0.4)])

ens.set_inserts('strings', [InsertFX('kl_reverb', mix=0.42, room=0.9, damp=0.6)])

def build_final():
    sc = Score()
    sc.from_ensemble(ens)
    for name, section in FINAL_ARRANGEMENT:
        base = name.rstrip('0123456789')
        knobs = dict(FINAL_KNOBS.get(name, FINAL_KNOBS.get(base, {})))
        chords = FINAL_PROGS.get(name, FINAL_PROGS.get(base))
        sc.append(section(chords=chords, time=FINAL_TIME, **knobs), name=name)
    return sc

final = build_final()

print(final)
play(final)
plot(final, animate=True)

Score(items=10, duration=67.826, tracks=['drums', 'bass', 'guitar', 'keys', 'strings'])
